# Glacier velocity reader — GrIMP / NSIDC

Pull **velocity time series for a region of interest (ROI)** straight from NSIDC, save them
as NetCDF, and plot them at a point.

Everything below is written as reusable functions so a run is reproducible from four inputs:

| Input | Meaning |
| --- | --- |
| `CENTER_XY` | Coordinate of the centre of the ROI |
| `DX_M`, `DY_M` | Distance in metres to add either side of the centre in *x* and *y* |
| `FIRST_DATE`, `LAST_DATE` | Date range to search at NSIDC |
| `PRODUCTS` | NSIDC product code(s), e.g. `NSIDC-0725`, `NSIDC-0481` |

Any of `CENTER_XY`, `DX_M`, `DY_M` and the output file name can instead be picked off a map —
see section 1b.

**Workflow**

0. `select_roi_interactive` — *(optional)* drag the ROI on a Greenland overview map when the
   coordinates are unknown, and get the glacier name suggested from the named flux gates.
1. `roi_bbox` — build a projected bounding box from the centre + distances.
2. `search_velocity_cogs` — query the NASA CMR catalog (`grimpfunc.get_urls`) for matching COGs.
3. `load_velocity_series` — lazily open them as a `nisardev.nisarVelSeries`.
4. `download_velocity_netcdfs` — download and write **one NetCDF per product type** (all bands).
5. `velocity_time_series` / `plot_velocity_time_series` — read the files back at a coordinate and plot.

**No-data convention.** GrIMP GeoTIFFs use sentinel values (`-2e9` for `vx`, `vy`, `dT`;
`-1` for `vv`, `ex`, `ey`, `ev`), and whether they are masked on read depends on whether the
COG declares `nodata` in its header. Speed makes this worse: `nisardev` derives `vv` as
`sqrt(vx² + vy²)`, so two unmasked sentinels produce `2.83e9` — a large *positive* number that
no threshold on speed can catch, and one that quietly becomes the maximum of your dataset.

Every array in this notebook passes through `normalize_nodata`, which thresholds each band
*and* makes `vv`/`ev` inherit the gaps of their components, so **NaN is the one and only
no-data value** on the way into a file, on the way back out, and in every plot.
`nodata_report` verifies it.

All coordinates are **EPSG:3413** (Polar Stereographic North, metres) unless stated otherwise.

## 0. Setup

The functions are in **`glacier_reader.py`** next to this notebook; the cells below only call
them. Requires `grimpfunc` and `nisardev` (from [github.com/fastice](https://github.com/fastice)),
plus rioxarray, rasterio, dask, pyproj and matplotlib - `pixi install` in this directory has all
of them.

Sections 2-10 document what each function does; section 11 runs the workflow.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Every function used below lives in glacier_reader.py next to this notebook.
import glacier_reader as gr
from glacier_reader import (
    DERIVED_BANDS, NSIDC_VELOCITY_PRODUCTS,
    describe_bbox, download_velocity_netcdfs, glacier_velocity_workflow,
    matlab_glacier_center, matlab_glacier_centers, nodata_report,
    plot_glacier_velocity, plot_velocity_time_series, read_velocity_netcdf,
    roi_bbox, summarize_search, velocity_time_series,
)

# Named scheduler + progress bar; NSIDC limits simultaneous connections.
gr.configure_dask(num_workers=2)

### NASA Earthdata login

Downloads need [Earthdata](https://urs.earthdata.nasa.gov) credentials. Run the cell, log in
once, and the cookie is reused (see the
[NSIDCLoginNotebook](https://github.com/fastice/GrIMPNotebooks/blob/master/NSIDCLoginNotebook.ipynb)).

In [ ]:
# Sets the GDAL cookie jar and returns the Earthdata login widget.
display(gr.login())

## 1. User inputs

The only cell that normally needs editing. It is tagged `parameters`, so the whole notebook can
also be swept over glaciers or date ranges with [papermill](https://papermill.readthedocs.io).

`CENTER_XY` is `(x, y)` in `CENTER_CRS`. For a geographic CRS (e.g. `4326`) pass it as
`(longitude, latitude)`.

In [ ]:
GLACIER_NAME = 'Jakobshavn'
CENTER_XY = (-183000.0, -2274200.0)   # centre of the ROI, on the fast trunk
CENTER_CRS = 3413                     # CRS of CENTER_XY; use 4326 for (lon, lat)
DX_M = 10000.0                        # metres to add either side in x
DY_M = 10000.0                        # metres to add either side in y

FIRST_DATE = '2018-01-01'             # NSIDC search start, YYYY-MM-DD
LAST_DATE = '2019-12-31'              # NSIDC search end, YYYY-MM-DD

# Annual mosaic, monthly mosaic and the TSX/TDX individual-glacier product
PRODUCTS = ['NSIDC-0725', 'NSIDC-0731', 'NSIDC-0481']

OUT_DIR = Path('Subsets')             # where the NetCDF files are written

### 1b. Don't know the ROI? Pick it off a map

If the glacier or its coordinates are not known in advance, skip the cell above and drag a box
on a Greenland overview instead: the most recent **annual velocity mosaic** (read decimated
straight from NSIDC, so it takes a few seconds, and cached afterwards) drawn over the standard
**BedMachine coastline**, with the 234 named flux gates of King et al. (2020) marked in red.

Needs an interactive backend — run `%matplotlib widget` first. The readout under the map shows
the box size, its centre, `dx`/`dy`, and the glacier it covers as you drag.

```python
%matplotlib widget
sel = gr.select_roi_interactive()
# drag a box, then:
CENTER_XY, DX_M, DY_M = sel.center, sel.dx_m, sel.dy_m
GLACIER_NAME = sel.prompt_name()     # confirm or override the suggested name
```

`sel.download(first_date=..., products=[...])` runs the whole workflow on the selection in one
step. The map alone is `gr.plot_greenland_overview()`, and the ROI → glacier lookup alone is
`gr.glaciers_in_roi(bbox)` / `gr.suggest_glacier_name(bbox)`.

In [ ]:
# Uncomment to pick the ROI from the map instead of typing coordinates above.
# Needs %matplotlib widget (inline figures cannot be dragged).

# %matplotlib widget
# sel = gr.select_roi_interactive()

In [ ]:
# ... then, after dragging a box on the map:
# CENTER_XY, DX_M, DY_M = sel.center, sel.dx_m, sel.dy_m
# CENTER_CRS = 3413
# GLACIER_NAME = sel.prompt_name()
# print(GLACIER_NAME, CENTER_XY, DX_M, DY_M)

## 2. Product registry and no-data conventions

`boxed` products (NSIDC-0481 TSX/TDX and NSIDC-0646 optical) are delivered as fixed ~50 km
tiles with their own names (e.g. `W69.10N`). One tile is one grid, so each tile gets its own
NetCDF file. The mosaics (0725/0727/0731/0766) are ice-sheet wide and are clipped to the ROI.

## 3. Region of interest

`roi_bbox` turns *centre coordinate + distance in x and y* into the `{'minx', 'miny', 'maxx',
'maxy'}` dictionary that `nisardev` and `rioxarray` expect. `bbox_lonlat` converts it to the
geographic bounding box the NASA CMR catalog searches with — the ROI edges are densified first,
because a rectangle in polar stereographic coordinates is not a rectangle in latitude/longitude.

## 4. Search the NSIDC catalog

`grimpfunc.get_urls` queries NASA CMR and returns one URL per band file. Those are collapsed
into **COG templates** — the file name with the band token replaced by `*`, which is the form
`nisardev` expects (it substitutes `vx`, `vy`, `ex`, ... in turn). Templates are grouped by
GrIMP box name, so an ROI straddling two TSX tiles yields two groups.

The product version is bumped up to four times if a search comes back empty, so the notebook
keeps working when NSIDC releases a new version.

## 5. Load a series and make NaN consistent

`load_velocity_series` opens the COGs lazily as a `nisardev.nisarVelSeries` — only the headers
and the ROI window are touched until the data are asked for.

`normalize_nodata` is the single place no-data is decided. It replaces every GrIMP sentinel
(and any `inf`) with NaN across all bands, makes the derived speed bands `vv`/`ev` inherit the
gaps of `vx`/`vy` and `ex`/`ey`, and drops the stale `_FillValue` coordinate that `nisardev`
carries with the sentinel values in it. It runs on the way into a NetCDF file **and** on the way
back out, so a file written by an older workflow is cleaned up the same way.

The inheritance step is not cosmetic: when a COG omits `nodata`, `vv` comes back as
`sqrt((-2e9)² + (-2e9)²) = 2.83e9`. It is positive, so it survives any sentinel threshold, and
it silently becomes the maximum of the dataset.

## 6. Save the gridded velocities to NetCDF

One file per product type — and per box for NSIDC-0481/0646, since each TSX/optical tile is its
own grid and cannot share a file with another. All bands go in: `vx`, `vy`, `vv`, `ex`, `ey`,
`ev` and, for the Sentinel-1 mosaics, `dT`.

The ROI, product code, box and date range are written into the file's global attributes, so a
file can be traced back to the call that made it.

NSIDC caps simultaneous connections, so a long download can die with an HTTP 404 or 503 on one
tile even when every file is present. `download_velocity_netcdfs` retries a product before
giving up (`retries=2`); if it still fails, lower `num_workers` rather than raising it.

## 7. Read the files back and build a time series at a point

`velocity_time_series` takes a directory (or a list of files) and a coordinate and returns a
tidy `DataFrame` — one row per granule per file, with `vx`, `vy`, `vv` and the error bands as
columns. Sampling is linear interpolation by default; use `method='nearest'` near the terminus
or an ice margin, where linear interpolation returns NaN as soon as one of the four surrounding
pixels is missing.

Every file is passed through `normalize_nodata` on read, so gaps are NaN regardless of how the
file was produced.

## 8. Plot the time series

Points are drawn only where the band is finite; the connecting line is broken wherever
consecutive observations are more than `max_gap_days` apart, so a data gap reads as a gap
rather than a straight line across it. Error bars use `ev`/`ex`/`ey` when the file has them,
and `show_span=True` adds horizontal bars for each granule's `time1`-`time2` window.

## 9. Check that no-data is consistent

`nodata_report` opens each file **without** cleaning it and counts NaNs and any surviving
sentinel values. Every `n_sentinel_left` should be `0`; `min` should be physically plausible
rather than `-2e9` or `-1`; and `max` for `vv` should be a real ice speed, not `2.8e9`.

## 10. Optional: ROI centres from MATLAB flux-gate files

The original notebook read glacier flux gates from MATLAB `.mat` files. Those still work as a
source of ROI centres — `matlab_glacier_center` returns the name and the centre of the gate,
which feeds straight into `download_velocity_netcdfs`.

In [ ]:
# Optional: pick the ROI centre from a MATLAB flux-gate file instead of typing coordinates.
# gates = matlab_glacier_centers('/Users/michalea/Documents/GIMP_Working/RECALC_discharge/redrawn_Jan25')
# GLACIER_NAME, CENTER_XY = matlab_glacier_center(gates.loc[0, 'file'])
# GLACIER_NAME, CENTER_XY

## 11. Run the workflow

### 11a. Check the ROI and see what is available

In [ ]:
roi = roi_bbox(CENTER_XY, DX_M, DY_M, center_crs=CENTER_CRS)
print(f'{GLACIER_NAME}: {describe_bbox(roi)}')
roi

In [ ]:
available = summarize_search(roi, FIRST_DATE, LAST_DATE, PRODUCTS)
available

### 11b. Download and save one NetCDF per product type

For NSIDC-0481/0646 this writes one file per GrIMP box the ROI touches. Re-running is cheap:
existing files are skipped unless `overwrite=True`.

In [ ]:
written = download_velocity_netcdfs(
    center=CENTER_XY,
    dx_m=DX_M,
    dy_m=DY_M,
    first_date=FIRST_DATE,
    last_date=LAST_DATE,
    products=PRODUCTS,
    out_dir=OUT_DIR,
    prefix=GLACIER_NAME,
    center_crs=CENTER_CRS,
    num_workers=2,
)
written

### 11c. Confirm the no-data convention

`n_sentinel_left` must be 0 for every band in every file, and each derived speed band must be
missing wherever its components are.

In [ ]:
report = nodata_report(written)
assert (report['n_sentinel_left'] == 0).all(), 'sentinel no-data values survived the write'

# vv is derived from vx, vy (and ev from ex, ey), so its gaps must cover theirs.
for name, group in report.groupby('file'):
    gaps = group.set_index('band')['nan_fraction']
    for derived, (band_x, band_y) in DERIVED_BANDS.items():
        if {derived, band_x, band_y} <= set(gaps.index):
            assert gaps[derived] >= gaps[band_x] - 1e-9, \
                f'{name}: {derived} is present where {band_x} is no-data'
report

### 11d. Time series at a point

Sampled at the ROI centre here; pass any coordinate (and `coord_crs=4326` for lon/lat).

`method='nearest'` is used because bilinear interpolation returns NaN as soon as *one* of the
four surrounding pixels is missing, which throws away observations near the terminus and along
the ice margin — exactly where a glacier time series is usually wanted. Switch to
`method='linear'` for a smoother sample well inside the ice.

In [ ]:
ts, ax = plot_glacier_velocity(
    written,
    *CENTER_XY,
    band='vv',
    coord_crs=CENTER_CRS,
    method='nearest',
    name=GLACIER_NAME,
    show_errors=True,
)
plt.tight_layout()
ts.head()

### 11e. Velocity components

The same frame holds `vx` and `vy`, so components come from one more call each.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for band, ax in zip(['vx', 'vy'], axes):
    plot_velocity_time_series(ts, band=band, ax=ax, title=f'{GLACIER_NAME} {band}')
axes[0].set_xlabel('')
plt.tight_layout()

### 11f. Map view for a single date

A sanity check that the ROI landed where you expected. The date with the most valid pixels is
chosen — picking an arbitrary one can land on a sparse granule and show mostly white.

In [ ]:
series = read_velocity_netcdf(written[0])

# The date with the largest valid fraction, so the map is not mostly no-data.
coverage = np.isfinite(series.subset.sel(band='vv')).mean(dim=('x', 'y'))
best_date = series.time[int(np.argmax(coverage.values))]

fig, ax = plt.subplots(figsize=(7, 6))
series.displayVelForDate(best_date, ax=ax, units='km', colorBarPosition='right')
ax.plot((roi['minx'] + roi['maxx']) / 2e3, (roi['miny'] + roi['maxy']) / 2e3,
        'w*', ms=15, mec='k', label='ROI centre')
ax.set_title(f'{GLACIER_NAME} speed, {best_date:%Y-%m-%d} '
             f'({float(coverage.max()) * 100:.0f}% valid)')
ax.legend(loc='lower left', fontsize=9)
plt.tight_layout()

### 11g. The same run as one call

`glacier_velocity_workflow` does 11a-11c in one call (ROI, catalog summary, download, no-data
check) and returns `{'bbox', 'available', 'files', 'report'}`. Use it when sweeping glaciers or
date ranges, or from a plain script with no notebook at all.

**Handing the files to ModelObsCompare.** These NetCDFs store bands on a `band` dimension;
`moc` labels the same axis `component`. `moc_py.obs_nsidc` bridges the two, so the files drop
straight into the model-vs-observation tools:

```python
import moc_py as mp
from moc_py.obs_nsidc import load_nsidc_netcdf

O   = load_nsidc_netcdf(out['files'][0], bands=['vv'])   # moc obs Dataset
roi = mp.roi_from_obs(O, margin=2000)
M   = mp.load_model('nw_fric1.nc')                       # from moc_export_model.m
D   = mp.spatial_diff(M, O, roi, time=2019.6)
mp.plot_diff(D, roi=roi)
```

In [ ]:
out = glacier_velocity_workflow(
    center=CENTER_XY,
    dx_m=DX_M,
    dy_m=DY_M,
    first_date=FIRST_DATE,
    last_date=LAST_DATE,
    products=PRODUCTS,
    out_dir=OUT_DIR,
    prefix=GLACIER_NAME,
    center_crs=CENTER_CRS,
)
out['files']

---

**Where the data went.** `OUT_DIR` now holds one `<glacier>_<product>[_<box>].nc` per product
type plus `<glacier>_roi.json` recording the exact inputs. Each file carries all bands, the ROI,
the date range and the product code in its attributes, and uses NaN - and only NaN - for
no-data.

**Reusing this elsewhere.** Everything described in sections 2-10 is in `glacier_reader.py`, so
other notebooks and scripts just `import glacier_reader as gr`. For the model-vs-observation
comparison, `moc_py.obs_nsidc` converts these files into the Dataset that `moc`'s `spatial_diff`,
`compare_timeseries`, ROI and variance tools expect - see section 11g.